In [7]:
!pip install sacrebleu

In [8]:
import pandas as pd
import sacrebleu

In [9]:
df2 = pd.read_csv("/kaggle/input/datasets/buketsak/data2-nmt-analysis-detailed-corrected/data2_nmt_analysis_detailed_corrected.csv")
df3 = pd.read_csv("/kaggle/input/datasets/buketsak/data3-nmt-analysis-n/data3_nmt_analysis.csv")

## Sacre-Blue Corpus

In [10]:
datasets = {'Dataset 2': df2, 'Dataset 3': df3}
model_columns = ['opus-tatoeba-en-tr', 'm2m100-418M', 'NLLB200-3.3B', 'translategemma-4b', 'llama3.1-IT-8B']

# Dictionary to store bleu scores
bleu_scores = {}

for name, df in datasets.items():
    bleu_scores[name] = {}
    references = [ref.lower() for ref in df['human_translations'].tolist()]  # list of reference sentences

    for model in model_columns:
        hypotheses = [hyp.lower() for hyp in df[model].tolist()]
        bleu = sacrebleu.corpus_bleu(hypotheses, [references])
        bleu_scores[name][model] = bleu.score

# Convert to DataFrame
bleu_df = pd.DataFrame(bleu_scores).round(2)
bleu_df

,Dataset 2,Dataset 3
opus-tatoeba-en-tr,43.56,21.77
m2m100-418M,23.85,8.12
NLLB200-3.3B,39.02,22.84
translategemma-4b,30.30,17.06
llama3.1-IT-8B,20.92,13.02


In [11]:
# loop over datasets
for name, df in datasets.items():
    # Create a df to store sentencelevel bleu
    sentence_bleu_df = pd.DataFrame()
    
    # Add reference column
    sentence_bleu_df['human_translations'] = df['human_translations']
    
    # Compute sentencelevel bleu for each model
    for model in model_columns:
        bleu_scorer = sacrebleu.metrics.BLEU(effective_order = True, tokenize='13a')
        scores = []
        for hyp, ref in zip(df[model].tolist(), df['human_translations'].tolist()):

            # Compute sentence level bleu
            score = bleu_scorer.sentence_score(hyp, [ref]).score
            scores.append(score)
        
        # Add scores as a new column
        sentence_bleu_df[model + '_bleu'] = scores

    filename = f"{name}_sentence_level_bleu.csv"
    sentence_bleu_df.to_csv(filename, index=False)
    print(f"Saved sentence-level BLEU for {name} to {filename}")

Saved sentence-level BLEU for Dataset 2 to Dataset 2_sentence_level_bleu.csv
Saved sentence-level BLEU for Dataset 3 to Dataset 3_sentence_level_bleu.csv


## NLTK, manual implementation (not to be used)

In [12]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [13]:
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import re

# Smoothing function for short sentences
smooth_fn = SmoothingFunction().method1

# Function to tokenize Turkish sentences (split punctuation)
def tokenize_turkish(sentence):
    # Separate punctuation from words
    # Keep Turkish characters intact
    tokens = re.findall(r"\w+|[^\w\s]", sentence, re.UNICODE) #either match a word or match punctuation
    return tokens

# Dictionary to store BLEU scores
bleu_scores = {}

for name, df in datasets.items():
    bleu_scores[name] = {}
    
    # Tokenize and lowercase all references
    references = [[tokenize_turkish(ref.lower())] for ref in df['human_translations'].tolist()]

    for model in model_columns:
        # Tokenize and lowercase all model outputs
        hypotheses = [tokenize_turkish(hyp.lower()) for hyp in df[model].tolist()]

        # Compute corpus-level BLEU
        score = corpus_bleu(references, hypotheses, smoothing_function=smooth_fn)
        bleu_scores[name][model] = score * 100  # Convert to percentage

# Convert to DataFrame for easy viewing
bleu_df = pd.DataFrame(bleu_scores).round(2)
bleu_df

,Dataset 2,Dataset 3
opus-tatoeba-en-tr,43.85,21.77
m2m100-418M,23.90,8.12
NLLB200-3.3B,39.36,22.84
translategemma-4b,30.02,17.03
llama3.1-IT-8B,21.09,12.93


## chr-F

In [14]:
# Datasets and model columns
datasets = {'Dataset 2': df2, 'Dataset 3': df3}
#model_columns = ['opus-tatoeba-en-tr', 'm2m100-418M', 'NLLB200-3.3B', 'translategemma-4b', 'llama3.1-IT-8B']

# store chrF scores in a dictionary
chrf_scores = {}

for name, df in datasets.items():
    chrf_scores[name] = {}
    
    # lowercase all 
    references = [ref.lower() for ref in df['human_translations'].tolist()]

    for model in model_columns:
        # lowercase all model outputs
        hypotheses = [hyp.lower() for hyp in df[model].tolist()]

        # compute corpus level chrF
        chrf = sacrebleu.corpus_chrf(hypotheses, [references])
        chrf_scores[name][model] = chrf.score  # chrF score in percentage

chrf_df = pd.DataFrame(chrf_scores).round(2)
chrf_df

,Dataset 2,Dataset 3
opus-tatoeba-en-tr,71.28,58.15
m2m100-418M,56.22,46.90
NLLB200-3.3B,66.12,59.48
translategemma-4b,62.84,54.50
llama3.1-IT-8B,51.88,49.39


## HUMAN EVALUATION

In [15]:
# Function to calculate dataset level human evaluation stats
def human_eval_stats(df, dataset_name):
    mean_score = df['evaluation_score'].mean()
    std_score = df['evaluation_score'].std()
    mean_score = round(mean_score, 2)
    std_score = round(std_score, 2)
    
    print(f"{dataset_name} - Average human score: {mean_score}, Std deviation: {std_score}")
    
    return mean_score, std_score

df2_mean, df2_std = human_eval_stats(df2, "Dataset 2")
df3_mean, df3_std = human_eval_stats(df3, "Dataset 3")

Dataset 2 - Average human score: 4.98, Std deviation: 0.14
Dataset 3 - Average human score: 4.91, Std deviation: 0.29


In [16]:
def human_eval_by_translator(df, dataset_name):
    # Group by translator
    stats = df.groupby('translator')['evaluation_score'].agg(['mean', 'std']).reset_index()
    stats['mean'] = stats['mean'].round(2)
    stats['std'] = stats['std'].round(2)
    
    print(f"\n{dataset_name} - Human evaluation scores by translator:")
    print(stats)
    
    return stats

df2_translator_stats = human_eval_by_translator(df2, "Dataset 2")
df3_translator_stats = human_eval_by_translator(df3, "Dataset 3")


Dataset 2 - Human evaluation scores by translator:
    translator  mean   std
0  Translator1  4.98  0.14
1  Translator2  4.98  0.14

Dataset 3 - Human evaluation scores by translator:
     translator  mean   std
0  Translator 1  4.91  0.29
1  Translator 2  4.91  0.29


## COMET

In [ ]:
pip uninstall transformers torchmetrics pytorch-lightning -y

In [ ]:
!pip install transformers==4.40.2
!pip install torchmetrics==0.11.4
!pip install pytorch-lightning==2.0.9
!pip install unbabel-comet

In [1]:
from comet import download_model, load_from_checkpoint

# download model
model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

hparams.yaml:   0%|          | 0.00/567 [00:00<?, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

Encoder model frozen.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:165: UserWarning: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
  rank_zero_warn(


In [2]:
model_columns = ['opus-tatoeba-en-tr', 'm2m100-418M', 'NLLB200-3.3B', 'translategemma-4b', 'llama3.1-IT-8B']

In [5]:
datasets = {'Dataset 2': df2, 'Dataset 3': df3}
comet_scores = {}

for name, df in datasets.items():
    comet_scores[name] = {}

    sources = df['source_sentence'].tolist()
    references = df['human_translations'].tolist()

    for model in model_columns:
        hypotheses = df[model].tolist()

        data = [
            {"src": src, "mt": mt, "ref": ref}
            for src, mt, ref in zip(sources, hypotheses, references)
        ]

        model_output = comet_model.predict(data, batch_size=16)

        comet_scores[name][model] = model_output.system_score

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Predicting DataLoader 0: 100%|██████████| 12/12 [00:02<00:00,  5.00it/s]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Predicting DataLoader 0: 100%|██████████| 12/12 [00:01<00:00,  6.83it/s]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Predicting DataLoader 0: 100%|██████████| 12/12 [00:01<00:00,  6.95it/s]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,

In [6]:
comet_df = pd.DataFrame(comet_scores).round(2)
comet_df

,Dataset 2,Dataset 3
opus-tatoeba-en-tr,0.94,0.86
m2m100-418M,0.90,0.78
NLLB200-3.3B,0.93,0.89
translategemma-4b,0.91,0.88
llama3.1-IT-8B,0.88,0.82
